In [2]:
%run Oncokb.ipynb

%run "/home/ubuntu/jupyter_notebooks/Internal/Cancer_hotspot/CH_display_table.ipynb"
    
%run "/home/ubuntu/jupyter_notebooks/Internal/Clinvar/Clinvar_annotation.ipynb"

%run "/home/ubuntu/jupyter_notebooks/Internal/Intogen/intogen_display_table.ipynb"
    
%run "getGeneInfoFromGeneCardsAndNCBI.ipynb"
    
%run "CKB_Core_Gene_Information_Extraction.ipynb"
    
%run "NCBI_gene_id.ipynb"

%run "dbsnp.ipynb"

In [13]:
import pandas as pd
import os
from datetime import datetime
import io
import shutil
import requests
import re
import functools 
import urllib.request
import xmltodict
from xmltodict import parse
import time
import requests
import numpy as np
from IPython.display import HTML, display
from ipywidgets import widgets, interact, Button, Layout, VBox, Output, Label, HTML, HBox
    
#User Input file upload path
USER_INPUT_FOLDER_PATH='/home/ubuntu/jupyter_notebooks/Internal/input_files/'
#Temporary foler path 
TEMP_FOLDER_PATH = '/home/ubuntu/jupyter_notebooks/Internal/input_files/Annotations_temporary/'
#SIFT Indel installation path
SIFT_INDEL_INSTALL_PATH='/home/ubuntu/in-silico_tools/SIFT'
SIFT_INDEL_DB_PATH='/home/ubuntu/in-silico_tools/SIFT/SIFT_INDEL_HG38'
#SIFT 4GPATH
SIFT_4G_INSTALL_PATH = '/home/ubuntu/in-silico_tools/SIFT_4G'
#dbNSFP installation path
dbNSFP_PATH='/home/ubuntu/in-silico_tools/dbNSFP'

Converts xlsx file to vcf file 

In [14]:
def convert_to_vcf(input_file, output_file):
    if input_file.endswith('.vcf'):
        # If the input file is already in VCF format, simply copy it to the output location
        shutil.copy(input_file, output_file)
        print(f"File '{input_file}' copied to '{output_file}'")
        return

    # Read the XLSX file into a pandas DataFrame
    df = pd.read_excel(input_file)

    # Extract CHROM and Position from 'Position (hg38)'
    df[['CHROM', 'POS']] = df['Position (hg38)'].str.extract(r'chr(\w+):(\d+)')

    # Create the VCF content
    vcf_content = '##fileformat=VCFv4.2\n'
    vcf_content += '#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\n'

    for _, row in df.iterrows():
        chrom = row['CHROM']
        pos = row['POS']
        ref = row['Ref']
        alt = row['Alt']
        info = row["VAF DNA RNA"]
        vcf_content += f'{chrom}\t{pos}\t.\t{ref}\t{alt}\t.\t.\t{info}\n'

    # Write the VCF content to a temporary file
    temp_vcf_file_path = 'temp_vcf_file.vcf'
    with open(temp_vcf_file_path, 'w') as temp_vcf_file:
        temp_vcf_file.write(vcf_content)

    # Sort the temporary VCF file based on chromosome positions
    with open(temp_vcf_file_path, 'r') as temp_vcf_file:
        lines = temp_vcf_file.readlines()

    # Remove header lines for sorting
    header_lines = lines[:2]
    vcf_content_to_sort = lines[2:]

    # Sort the VCF content
    sorted_vcf_content = ''.join(sorted(vcf_content_to_sort, key=lambda x: (x.split('\t')[0], int(x.split('\t')[1]))))

    # Write the sorted VCF content to the output file
    with open(output_file, 'w') as vcf_file:
        vcf_file.write(''.join(header_lines) + sorted_vcf_content)

    # Remove temporary VCF file
    os.remove(temp_vcf_file_path)
    print(f"VCF file saved to '{output_file}'")

# Provide your input and output file paths
#input_file_path = 'somatic_snv_and_indel_with_new_variants.xlsx'
#input_file_path = 'Somatic_SNV.vcf_processed_15_var.vcf'
input_file_path = 'Somatic_SNV_with_pathogenic_var.vcf_cgp_snv_indel.vcf'
input_filename = os.path.basename(input_file_path)
input_filename = os.path.splitext(input_filename)[0]
output_file_name=input_filename + '.vcf'
output_file_path = os.path.join(USER_INPUT_FOLDER_PATH+ input_filename + '.vcf')

# Convert the XLSX file to VCF and sort based on chromosome
#convert_to_vcf(input_file_path, output_file_path)


The VCF saved is read and tools like SIFT INDEL, SIFT 4G, dNSFP and CADD-SNV is included. These tools take input as the VCF data 

In [15]:
def read_vcf(input_file):
    try:
        with open(input_file, 'r', encoding='latin-1') as f:
            lines = [l for l in f if not l.startswith('##')]
        '''
        # Debug: Print sample lines after filtering
        print("Sample lines after filtering:")
        for line in lines[:5]:  # Print the first 5 lines for inspection
            print(line)
        '''
        data_frame = pd.read_csv(
            io.StringIO(''.join(lines)),
            dtype={'#CHROM': str, 'POS': int, 'ID': str, 'REF': str, 'ALT': str,
                   'QUAL': str, 'FILTER': str, 'INFO': str},
            sep='\t'
        )
        
        def extract_dp(text):
          for part in text.split(';'):
            k, v = part.strip().split('=')
            if k == 'DP':
              return int(v)  # Assuming DP value is an integer
          return None  # Return None if DP not found

        # Apply the function to extract DP values and create a new column
        data_frame['Depth'] = data_frame['INFO'].apply(extract_dp)
        data_frame['Allele Frequency (%)'] = (data_frame['HCC1187BL'].str.split(':', expand=True)[3])
        data_frame['Allele Frequency (%)']=data_frame['Allele Frequency (%)'].astype(float)
        data_frame['Allele Frequency (%)']=(data_frame['Allele Frequency (%)']*100)
        if data_frame['#CHROM'].str.contains('chr').any():
            data_frame['#CHROM'] = data_frame['#CHROM'].str.replace('chr', '')
        if data_frame['#CHROM'].str.contains('Chr').any():
            data_frame['#CHROM'] = data_frame['#CHROM'].str.replace('Chr', '')
        
        #data_frame.rename(columns={'INFO': 'Allele Frequency (%)'}, inplace=True)
        #data_frame['Allele Frequency (%)'] = (data_frame['Allele Frequency (%)'] * 100).astype(int)
        data_frame.drop_duplicates(subset=['POS'],inplace = True)
        
    
        return data_frame
    
    except Exception as e:
        print(f"Error reading VCF file: {e}")
        return None
#display(read_vcf('input_files/somatic_snv_and_indel_with_new_variants.vcf'))
#display(read_vcf('input_files/Somatic_SNV.vcf_processed_15_var.vcf'))
#inp_df = read_vcf('input_files/Somatic_SNV.vcf_processed_15_var.vcf')
# inp_df = read_vcf('input_files/Somatic_SNV.vcf_processed.vcf')
inp_df = read_vcf(USER_INPUT_FOLDER_PATH+input_file_path)
display(inp_df)

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,HCC1187BL,HCC1187C,Gene name_BED file,Depth,Allele Frequency (%)
0,1,9665241,.,A,G,.,PASS,DP=130;MQ=185.80;FractionInformativeReads=0.977,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:47,0:0.000:24,0:23,0:47:.:.","0/1:71.13:54,26:0.325:32,11:22,15:80:28,26,18,...",PIK3CD,130,0.0
1,1,11216105,.,T,A,.,PASS,DP=155;MQ=198.79;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:-0.00:57,0:0.000:30,0:27,0:57:.:.","0/1:205.89:0,98:1.000:0,47:0,51:98:0,0,43,55:0...",MTOR,155,0.0
2,1,64931096,.,A,T,.,PASS,DP=206;MQ=105.45;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:50,0:0.000:25,0:25,0:50:.:.","0/1:126.57:81,75:0.481:33,36:48,39:156:46,35,3...",JAK1,206,0.0
3,1,64969201,.,G,GT,.,PASS,DP=210;MQ=214.69;FractionInformativeReads=0.886,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:45,0:0.000:18,0:27,0:45:.:.","0/1:110.62:74,67:0.475:34,37:40,30:141:35,39,2...",JAK1,210,0.0
4,1,65022230,.,G,A,.,PASS,DP=243;MQ=214.83;FractionInformativeReads=0.996,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:-0.00:62,0:0.000:31,0:31,0:62:.:.","0/1:132.48:105,75:0.417:56,39:49,36:180:45,60,...",JAK1,243,0.0
5,1,97142697,.,T,G,.,PASS,DP=116;MQ=190.00;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:-0.00:49,0:0.000:21,0:28,0:49:.:.","0/1:181.80:0,67:1.000:0,36:0,31:67:0,0,38,29:0...",DPYD,116,0.0
6,1,97177779,.,T,A,.,PASS,DP=116;MQ=195.08;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:40,0:0.000:19,0:21,0:40:.:.","0/1:92.37:41,35:0.461:18,14:23,21:76:13,28,14,...",DPYD,116,0.0
7,1,97371462,.,G,A,.,PASS,DP=109;MQ=190.06;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:46,0:0.000:20,0:26,0:46:.:.","0/1:102.48:33,30:0.476:18,9:15,21:63:15,18,19,...",DPYD,109,0.0
8,1,97527794,.,GA,G,.,PASS,DP=132;MQ=193.39;FractionInformativeReads=0.780,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:32,2:0.059:17,1:15,1:34:.:.","0/1:56.78:31,38:0.551:14,20:17,18:69:15,16,15,...",DPYD,132,5.9
9,1,156865995,.,A,G,.,PASS,DP=238;MQ=208.22;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:72,0:0.000:38,0:34,0:72:.:.","0/1:27.68:158,8:0.048:73,2:85,6:166:88,70,3,5:...",NTRK1,238,0.0


In [16]:
import os

def SIFT_indel(data_frame, TEMP_FOLDER_PATH, USER_INPUT_FOLDER_PATH, input_filename, SIFT_INDEL_INSTALL_PATH, SIFT_INDEL_DB_PATH):
    """
    This function processes indel (insertion-deletion) mutations from a DataFrame,
    formats them for the SIFT Indel tool, and executes the SIFT Indel command.
    
    Args:
    - data_frame: pandas DataFrame containing VCF data
    - TEMP_FOLDER_PATH: str, path to temporary folder
    - USER_INPUT_FOLDER_PATH: str, path to user input folder
    - input_filename: str, name of the input file
    - SIFT_INDEL_INSTALL_PATH: str, path to SIFT Indel installation
    - SIFT_INDEL_DB_PATH: str, path to SIFT Indel database
    """
    
    # Extract relevant columns from the data frame
    ref = list(data_frame['REF'])
    alt = list(data_frame['ALT'])
    pos = list(data_frame['POS'])
    chrom = list(data_frame['#CHROM'])
    
    # Initialize lists for storing indel positions and sequences
    indel_list_pos = []
    indel_list_chrom = []
    indel_list_ref = []
    indel_list_alt = []

    # Identify indels (differences in length between ref and alt, with no commas in alt)
    for i in range(len(ref)):
        if len(ref[i]) != len(alt[i]) and ',' not in alt[i]:
            indel_list_pos.append(pos[i])
            indel_list_chrom.append(chrom[i])
            indel_list_ref.append(ref[i])
            indel_list_alt.append(alt[i])

    # Separate indels into insertions and deletions
    insertion_list_pos = []
    insertion_list_ref = []
    insertion_list_alt = []
    insertion_list_chrom = []

    deletion_list_chrom = []
    deletion_list_pos = []
    deletion_list_ref = []
    deletion_list_alt = []

    for i in range(len(indel_list_ref)):
        if len(indel_list_ref[i]) < len(indel_list_alt[i]):
            insertion_list_chrom.append(indel_list_chrom[i])
            insertion_list_pos.append(indel_list_pos[i])
            insertion_list_ref.append(indel_list_ref[i])
            insertion_list_alt.append(indel_list_alt[i])
        else:
            deletion_list_chrom.append(indel_list_chrom[i])
            deletion_list_pos.append(indel_list_pos[i])
            deletion_list_ref.append(indel_list_ref[i])
            deletion_list_alt.append(indel_list_alt[i])

    # Helper function to split a word into a list of characters
    def split(word):
        return [char for char in word]

    # Format deletions for SIFT Indel
    sift_format_deletion = []
    for i in range(len(deletion_list_ref)):
        sub = len(deletion_list_ref[i]) - len(deletion_list_alt[i])
        del_ref = split(deletion_list_ref[i])
        del_alt = split(deletion_list_alt[i])
        for j in range(len(del_alt)):
            del_ref.remove(del_alt[j])

        pos2 = deletion_list_pos[i] + sub
        sift_format_deletion.append(deletion_list_chrom[i] + ',' + str(deletion_list_pos[i]) + ',' + str(pos2) + ',' + '1' + ',' + '/')

    # Format insertions for SIFT Indel
    inserts = []
    sift_format_insertion = []

    for i in range(len(insertion_list_alt)):
        ins_strings = ''
        sub = len(insertion_list_alt[i]) - len(insertion_list_ref[i])
        ins_ref = split(insertion_list_ref[i])
        ins_alt = split(insertion_list_alt[i])
        for j in range(len(ins_ref)):
            ins_alt.remove(ins_ref[j])
        for k in range(len(ins_alt)):
            ins_strings += ins_alt[k]

        sift_format_insertion.append(insertion_list_chrom[i] + ',' + str(insertion_list_pos[i]) + ',' + str(insertion_list_pos[i]) + ',' + '1' + ',' + ins_strings)

    # Combine formatted insertions and deletions
    final_list = sift_format_insertion + sift_format_deletion

    # Write the formatted indels to a file for SIFT Indel processing
    sift_input_file = TEMP_FOLDER_PATH + "indel_file_" + input_filename + ".txt"
    with open(sift_input_file, "w") as textfile:
        for element in final_list:
            textfile.write(element + "\n")
    
    # Define output file paths for the SIFT Indel process
    sift_process_output = USER_INPUT_FOLDER_PATH + input_filename + "_sift_indel_process_out.txt"
    sift_prediction_file = USER_INPUT_FOLDER_PATH + 'SIFT_indel_' + input_filename + '.tsv'
    SIFTIndel_out = USER_INPUT_FOLDER_PATH + input_filename + '_SIFTIndel_out.txt'
    SIFTIndel_err = USER_INPUT_FOLDER_PATH + input_filename + '_SIFTIndel_err.txt'

    # Construct the command to run the SIFT Indel tool
    command = f'nohup sh SIFT_indel.sh {SIFT_INDEL_INSTALL_PATH} {sift_input_file} {SIFT_INDEL_DB_PATH} {TEMP_FOLDER_PATH} {sift_process_output} {sift_prediction_file} {USER_INPUT_FOLDER_PATH} > {SIFTIndel_out} 2> {SIFTIndel_err} &'
    
    # Execute the command
    os.system(command)
        

'''
def SIFT_indel(input_file):
    # data_frame=read_vcf(USER_INPUT_FOLDER_PATH+input_file)
    data_frame=input_file
    ref=list(data_frame['REF'])
    alt=list(data_frame['ALT'])
    pos=list(data_frame['POS'])
    chrom=list(data_frame['#CHROM'])

    indel_list_pos=[]
    indel_list_chrom=[]
    indel_list_ref=[]
    indel_list_alt=[]

    for i in range(len(ref)):
        if len(ref[i])!=len(alt[i])and ','not in alt[i]:
            indel_list_pos.append(pos[i])
            indel_list_chrom.append(chrom[i])
            indel_list_ref.append(ref[i])
            indel_list_alt.append(alt[i])

    insertion_list_pos=[]
    insertion_list_ref=[]
    insertion_list_alt=[]
    insertion_list_chrom=[]

    deletion_list_chrom=[]
    deletion_list_pos=[]
    deletion_list_ref=[]
    deletion_list_alt=[]

    for i in range(len(indel_list_ref)):
        if len(indel_list_ref[i])< len(indel_list_alt[i]):
            insertion_list_chrom.append(indel_list_chrom[i])
            insertion_list_pos.append(indel_list_pos[i])
            insertion_list_ref.append(indel_list_ref[i])
            insertion_list_alt.append(indel_list_alt[i])
        else:
            deletion_list_chrom.append(indel_list_chrom[i])
            deletion_list_pos.append(indel_list_pos[i])
            deletion_list_alt.append(indel_list_alt[i])
            deletion_list_ref.append(indel_list_ref[i])

    def split(word):
        return [char for char in word]

    sift_format_deletion=[]
    for i in range(len(deletion_list_ref)):
        sub=len(deletion_list_ref[i])-len(deletion_list_alt[i])
        del_ref=split(deletion_list_ref[i])
        del_alt=split(deletion_list_alt[i])
        for j in range(len(del_alt)):
            del_ref.remove(del_alt[j])

        pos2=deletion_list_pos[i]+sub
        sift_format_deletion.append(deletion_list_chrom[i]+','+str(deletion_list_pos[i])+','+str(pos2)+','+'1'+','+'/')

    inserts=[]
    sift_format_insertion=[]

    for i in range(len(insertion_list_alt)):
        ins_strings=''
        sub=len(insertion_list_alt[i])-len(insertion_list_ref[i])
        ins_ref=split(insertion_list_ref[i])
        ins_alt=split(insertion_list_alt[i])
        for j in range(len(ins_ref)):
            ins_alt.remove(ins_ref[j])
        for k in range(len(ins_alt)):
            ins_strings=ins_strings+ins_alt[k]
        
        sift_format_insertion.append(insertion_list_chrom[i]+','+str(insertion_list_pos[i])+','+str(insertion_list_pos[i])+','+'1'+','+ins_strings)

    final_list=sift_format_insertion+sift_format_deletion

    #input file prepared for SIFT Indel process
    sift_input_file = TEMP_FOLDER_PATH+"indel_file_"+ input_filename +".txt" 
    textfile = open(sift_input_file, "w")
    for element in final_list:
        textfile.write(element + "\n")
    textfile.close()
    
    #path="indel_file"+file+".txt"
    sift_process_output=USER_INPUT_FOLDER_PATH+input_filename+"_sift_indel_process_out.txt"
    sift_prediction_file=USER_INPUT_FOLDER_PATH+'SIFT_indel'+'_'+input_filename+'.tsv'
    SIFTIndel_out = USER_INPUT_FOLDER_PATH+input_filename+'_SIFTIndel_out.txt'
    SIFTIndel_err = USER_INPUT_FOLDER_PATH+input_filename+'_SIFTIndel_err.txt'
    
    command='nohup sh SIFT_indel.sh '+SIFT_INDEL_INSTALL_PATH +' '+sift_input_file+' '+SIFT_INDEL_DB_PATH +' '+TEMP_FOLDER_PATH +' '+ sift_process_output + ' ' + sift_prediction_file +' '+USER_INPUT_FOLDER_PATH+' >'+SIFTIndel_out + ' 2>'+SIFTIndel_err +' &'
    #print("Command :",command)
    r1=os.system(command)
'''

'\ndef SIFT_indel(input_file):\n    # data_frame=read_vcf(USER_INPUT_FOLDER_PATH+input_file)\n    data_frame=input_file\n    ref=list(data_frame[\'REF\'])\n    alt=list(data_frame[\'ALT\'])\n    pos=list(data_frame[\'POS\'])\n    chrom=list(data_frame[\'#CHROM\'])\n\n    indel_list_pos=[]\n    indel_list_chrom=[]\n    indel_list_ref=[]\n    indel_list_alt=[]\n\n    for i in range(len(ref)):\n        if len(ref[i])!=len(alt[i])and \',\'not in alt[i]:\n            indel_list_pos.append(pos[i])\n            indel_list_chrom.append(chrom[i])\n            indel_list_ref.append(ref[i])\n            indel_list_alt.append(alt[i])\n\n    insertion_list_pos=[]\n    insertion_list_ref=[]\n    insertion_list_alt=[]\n    insertion_list_chrom=[]\n\n    deletion_list_chrom=[]\n    deletion_list_pos=[]\n    deletion_list_ref=[]\n    deletion_list_alt=[]\n\n    for i in range(len(indel_list_ref)):\n        if len(indel_list_ref[i])< len(indel_list_alt[i]):\n            insertion_list_chrom.append(indel

In [24]:
# def SIFT_indel(inp_df):
#     #print(data_frame)
#     chrom = list(inp_df['#CHROM'].astype(str))
#     pos = list(inp_df['POS'].astype(str))
#     ref = list(inp_df['REF'].astype(str))
#     alt = list(inp_df['ALT'].astype(str))

#     indel_list_pos = []
#     indel_list_chrom = []
#     indel_list_ref = []
#     indel_list_alt = []
    
#     for i in range(len(ref)):
#         if len(ref[i])!=len(alt[i])and ','not in alt[i]:
#             indel_list_pos.append(pos[i])
#             indel_list_chrom.append(chrom[i])
#             indel_list_ref.append(ref[i])
#             indel_list_alt.append(alt[i])
#     #print('Indel list')
#     #print(indel_list_pos, indel_list_chrom, indel_list_ref, indel_list_alt)
        
#     insertion_list_pos=[]
#     insertion_list_ref=[]
#     insertion_list_alt=[]
#     insertion_list_chrom=[]

#     deletion_list_chrom=[]
#     deletion_list_pos=[]
#     deletion_list_ref=[]
#     deletion_list_alt=[]

#     for i in range(len(indel_list_ref)):
#         if len(indel_list_ref[i])< len(indel_list_alt[i]):
#             insertion_list_chrom.append(indel_list_chrom[i])
#             insertion_list_pos.append(indel_list_pos[i])
#             insertion_list_ref.append(indel_list_ref[i])
#             insertion_list_alt.append(indel_list_alt[i])
#         else:
#             deletion_list_chrom.append(indel_list_chrom[i])
#             deletion_list_pos.append(indel_list_pos[i])
#             deletion_list_alt.append(indel_list_alt[i])
#             deletion_list_ref.append(indel_list_ref[i])
#     #print('Insertion list')
#     #print(insertion_list_pos, insertion_list_ref, insertion_list_alt, insertion_list_chrom)
    
#     #print('Deletion list')
#     #print(deletion_list_chrom, deletion_list_pos, deletion_list_ref, deletion_list_alt)
#     def split(word):
#         return [char for char in word]
    
#     sift_format_deletion=[]
#     for i in range(len(deletion_list_ref)):
#         sub = str(len(deletion_list_ref[i]) - len(deletion_list_alt[i]))
#         #print(sub)
#         del_ref=split(deletion_list_ref[i])
#         del_alt=split(deletion_list_alt[i])
#         for j in range(len(del_alt)):
#             del_ref.remove(del_alt[j])

#         pos2=deletion_list_pos[i]+sub
#         sift_format_deletion.append(deletion_list_chrom[i]+','+str(deletion_list_pos[i])+','+str(pos2)+','+'1'+','+'/')
#     #print('SIFT deletion')
#     #print(sift_format_deletion)
    
#     inserts=[]
#     sift_format_insertion=[]
#     for i in range(len(insertion_list_alt)):
#         ins_strings=''
#         sub=len(insertion_list_alt[i])-len(insertion_list_ref[i])
#         ins_ref=split(insertion_list_ref[i])
#         ins_alt=split(insertion_list_alt[i])
#         for j in range(len(ins_ref)):
#             ins_alt.remove(ins_ref[j])
#         for k in range(len(ins_alt)):
#             ins_strings=ins_strings+ins_alt[k]
        
#         sift_format_insertion.append(insertion_list_chrom[i]+','+str(insertion_list_pos[i])+','+str(insertion_list_pos[i])+','+'1'+','+ins_strings)
#     #print('SIFT Insertion')
#     #print(sift_format_insertion)
    
#     final_list=sift_format_insertion+sift_format_deletion
#     sift_input_file = TEMP_FOLDER_PATH + "indel_file_" + input_filename + ".txt"
#     textfile = open(sift_input_file, "w")
#     for element in final_list:
#         textfile.write(element + "\n")
#     textfile.close()
#     sift_process_output = USER_INPUT_FOLDER_PATH + input_filename + "_sift_indel_process_out.txt"
#     sift_prediction_file = os.path.join(USER_INPUT_FOLDER_PATH, 'SIFT_indel' + '_' + input_filename + '.tsv')
#     SIFTIndel_out = USER_INPUT_FOLDER_PATH + input_filename + '_SIFTIndel_out.txt'
#     SIFTIndel_err = USER_INPUT_FOLDER_PATH + input_filename + '_SIFTIndel_err.txt'
#     #print(sift_prediction_file)
#     command='nohup sh SIFT_indel.sh '+SIFT_INDEL_INSTALL_PATH +' '+sift_input_file+' '+SIFT_INDEL_DB_PATH +' '+TEMP_FOLDER_PATH +' '+ sift_process_output + ' ' + sift_prediction_file +' '+USER_INPUT_FOLDER_PATH+' >'+SIFTIndel_out + ' 2>'+SIFTIndel_err +' &'
#     #print("Command :",command)
#     r1=os.system(command)

def SIFT4G_1(inp_df):
    #print('inside SIFT 4G')
    print(inp_df)
    SIFT4G_out = USER_INPUT_FOLDER_PATH+input_filename+'_SIFT4G_out.txt'
    SIFT4G_err = USER_INPUT_FOLDER_PATH+input_filename+'_SIFT4G_err.txt'
    SIFT4G_annotation_file = os.path.join(TEMP_FOLDER_PATH+input_filename+'_SIFTannotations.xls')
    #print(SIFT4G_annotation_file)
    SIFT4G='nohup sh SIFT_4G.sh '+SIFT_4G_INSTALL_PATH +' '+USER_INPUT_FOLDER_PATH+' '+ output_file_name +' '+TEMP_FOLDER_PATH+' '+SIFT4G_annotation_file+' >'+SIFT4G_out + ' 2>'+SIFT4G_err +' &'
    r=os.system(SIFT4G)
 
def dbNSFP_1(inp_df):
    dbnsfp_out = USER_INPUT_FOLDER_PATH+input_filename+'_dbNSFP_out.txt'
    dbnsfp_err = USER_INPUT_FOLDER_PATH+input_filename+'_dbNSFP_err.txt'
    db='nohup sh dbNSFP.sh '+dbNSFP_PATH +' '+USER_INPUT_FOLDER_PATH+ output_file_name+' >'+dbnsfp_out + ' 2>'+dbnsfp_err +' &'
    #print(db)
    result=os.system(db)

def fetch_data_from_cadd(url):
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Failed to fetch data from {url}")
        return None
    
def CADD_SNV(inp_df):
    output_data = []
    for index, row in inp_df.iterrows():
        chrom = row['#CHROM']
        pos = row['POS']
        alt = row['ALT']
        ref = row['REF']
        url = f'https://cadd.gs.washington.edu/api/v1.0/GRCh38-v1.7_inclAnno/{chrom}:{pos}_{ref}_{alt}'
        #print(f'Processing: {url}')
        response_data = fetch_data_from_cadd(url)
        if response_data:
            for item in response_data:
                item['Chrom'] = chrom
                item['Position'] = pos
                item['Ref'] = ref
                item['Alt'] = alt
            output_data.extend(response_data)
    return output_data

def clinvar_extraction(input_df,gref):
    #making a copy of the input file dataframe to add the intermediate processes columns
    input_df_copy=input_df.copy()
    dictionary_df=pd.DataFrame(columns=["Locus:ref:alt"])

    for index, row in input_df_copy.iterrows():
        output_df=pd.DataFrame()
        chr_no = row['#CHROM']
        chr_pos = row['POS']
        #coding = row["coding"]
        ref = row['REF']
        alt = row['ALT']
        check=f"{chr_no}:{chr_pos}:{ref}:{alt}"
        #print (check)
        #print(list(dictionary_df["Locus:ref:alt"]))
        if check in list(dictionary_df["Locus:ref:alt"]):
            #print("using cached data")
            cached_row = dictionary_df[dictionary_df["Locus:ref:alt"] == check].iloc[0]
            # Update input_df with cached data
            input_df.loc[index, 'Gene'] = cached_row['Gene']
            input_df.loc[index, 'Clinvar_accession_no'] = cached_row['Clinvar_accession_no']
            input_df.loc[index, 'Clinvar_Protein_change'] = cached_row['ClinVar_Protein_change']
            input_df.loc[index, 'gnomAD_value'] = cached_row['gnomAD_value']
            input_df.loc[index, '1000_Genomes_value'] = cached_row['1000_Genomes_value']
            input_df.loc[index, 'ExAC_value'] = cached_row['ExAC_value']
            input_df.loc[index, 'dbSNP_id'] = cached_row['dbSNP_id']
            input_df.loc[index, 'OMIM_id'] = cached_row['OMIM_id']
            input_df.loc[index, 'Germline_classification'] = cached_row['Germline_classification']
            input_df.loc[index, 'Somatic_clinical_impact'] = cached_row['Somatic_clinical_impact']
            input_df.loc[index, 'Clinvar_link'] = cached_row['Clinvar_link']
        else:
            if len(ref)==len(alt)==1:
                output_df=clinvar_ref_alt_input(chr_no,chr_pos,ref,alt,gref)
                output_df.loc[0,"Locus:ref:alt"]= f"{chr_no}:{chr_pos}:{ref}:{alt}"
                dictionary_df=pd.concat([dictionary_df, output_df], axis=0)
                #display(dictionary_df)
            else:
                ref=ref.strip()
                alt=alt.strip()
                output_df= clinvar_ref_alt_input_indel(chr_no,chr_pos,ref,alt,gref)
                output_df.loc[0,"Locus:ref:alt"]= f"{chr_no}:{chr_pos}:{ref}:{alt}"
                dictionary_df=pd.concat([dictionary_df, output_df], axis=0)
            if not output_df.empty:
                #Adding the clinvar annotation columns to the original input file dataframe
                input_df.loc[index, 'Gene'] = output_df.loc[0,'Gene']
                input_df.loc[index, 'Clinvar_accession_no'] = output_df.loc[0,'Clinvar_accession_no']
                input_df.loc[index, 'Clinvar_Protein_change'] = output_df.loc[0,'ClinVar_Protein_change']
                input_df.loc[index, 'gnomAD_value'] = output_df.loc[0,'gnomAD_value']
                input_df.loc[index, '1000_Genomes_value'] = output_df.loc[0,'1000_Genomes_value']
                input_df.loc[index, 'ExAC_value'] = output_df.loc[0,'ExAC_value']
                input_df.loc[index, 'dbSNP_id'] = output_df.loc[0,'dbSNP_id']
                input_df.loc[index, 'OMIM_id'] = output_df.loc[0,'OMIM_id']
                input_df.loc[index, 'Germline_classification'] = output_df.loc[0,'Germline_classification']
                input_df.loc[index, 'Somatic_clinical_impact'] = output_df.loc[0,'Somatic_clinical_impact']
                input_df.loc[index, 'Clinvar_link'] = output_df.loc[0,'Clinvar_link']
    input_df['Clinvar_Protein_change']= input_df['Clinvar_Protein_change'].fillna("")
    return input_df

def dbsnp_extraction(input_df,gref):
    """ Function for extraction of dnsnp id and gnomad-genomes-global frequecy from the dbsnp database
    Args: input_df: the input dataframe with the chromosome no, chroomosome position,reference allele and alternate allele columns
    gref: reference genome "37" or "38"
    returns: the input dataframe with the dbSNP_Id and gnomAD-genomes columns added to it
    """
    #define the column names in the input dataframe
    chr_no = '#CHROM'
    chr_pos = 'POS'
    ref = 'REF'
    alt = 'ALT'
    vtype = "vtype"
    #process the columns required for constructing the input query for dbsnp and matching with the database
    input_df= input_dataframe_processing(input_df,ref,alt)
    display(input_df)
    #iterate through the rows of the input dataframe and fetch the dbsnp data
    for index, row in input_df.iterrows():
        chr_no_row = row[chr_no]
        chr_pos_row = row[chr_pos]
        ref_row = row[ref+"_dbsnp ref"]
        alt_row = row[alt+"_dbsnp alt"]
        vtype_row=row["vtype"]
        dbsnp_id,freq=dbsnp(chr_no_row,chr_pos_row,ref_row,alt_row,vtype_row,"38")#function call to extract from the database
        #populate input dataframe with the values
        input_df.at[index, "dbSNP_rs_id"] = dbsnp_id
        input_df.at[index, "gnomAD_value_dbSNP"] = freq
    #drop extra columns generated while processing 
    input_df = input_df.drop([vtype,ref+"_dbsnp ref",alt+"_dbsnp alt"], axis=1)
    # display(input_df)
    #return the output
    return input_df
    
    
# Example usage for processing a single VCF file
#inp_df = read_vcf(output_file_path)


In [25]:
def getCKBCoreData(input_df):
    gene_id_dict={}#dict for storing extracted ncbi_id
    ckb_dict={}#dict for storing extracted ckb core data
    gcolumn="Gene name_BED file"# Name of the column with gene names
    input_df[gcolumn]=input_df[gcolumn].fillna("") #fill empty rows in gene name column
    if gcolumn in input_df.columns:
        for i,row in input_df.iterrows():
            gname =row[gcolumn]
            print(gname)
            if gname is not "":
                if gname in gene_id_dict.keys(): # accessing stored ckb core data from previous parsing
                    ncbi_id=gene_id_dict[gname]
                    ckb_data=ckb_dict[gname]
                    print(f"Using cached data for {gname}")
                else:
                    ncbi_id = get_gene_id(gname) #calling the function to get gene id from NCBI
                    gene_id_dict[gname]=ncbi_id
                    if ncbi_id is not None:
                        ckb_data = getCKBGeneInfoForGeneID(ncbi_id) # calling funciton to fetch ckb core information
                        ckb_dict[gname]=ckb_data #storing the data in the dict
                    else:
                        ckb_dict[gname]={}  
                print(ncbi_id)
                print(ckb_data)
                #check if ncbi_id was extracted and populate input dataframe new columns
                if ncbi_id is not None:
                    input_df.loc[i, 'NCBI Gene ID'] = ncbi_id
                    #check if ckb core data is availiable
                    if len(ckb_data)!=0:
                        input_df.loc[i, 'CKB Core _Synonyms'] = ckb_data['Synonyms']
                        input_df.loc[i, 'CKB Core_GeneRole'] = ckb_data['Gene Role']
                        input_df.loc[i, 'CKB Core_Gene_Description'] = ckb_data['Gene Description']
                    else:
                        print(f"No data in CKB core for {gname}-{ncbi_id}")
                
    else:
        raise Exception("getCKBCoreData:'Gene name_BED file' column expected in input. Aborting....")
        
    return input_df 

In [26]:
# %run "getGeneInfoFromGeneCardsAndNCBI.ipynb"
# def getCKBCoreData(input_df):
#     if 'Gene name_BED file' in input_df.columns:
#         for i,row in input_df.iterrows():
#             gname =row['Gene name_BED file']
#             gc_name,alias,ncbi_id = getGeneCardsData(gname)
#             input_df.loc[i, 'NCBI Gene ID'] = ncbi_id
#             ckb_data = getCKBGeneInfoForGeneID(ncbi_id)
#             display(ckb_data)
#             input_df.loc[i, 'CKB Core _Synonyms'] = ckb_data['Synonyms']
#             input_df.loc[i, 'CKB Core_GeneRole'] = ckb_data['Gene Role']
#             input_df.loc[i, 'CKB Core_Gene_Description'] = ckb_data['Gene Description']            
#     else:
#         raise Exception("getCKBCoreData:'Gene name_BED file' column expected in input. Aborting....")
        
#     return input_df    
'''
ckb_output = getCKBCoreData(inp_df)
if not ckb_output.empty:
    output_files_path=os.path.join(USER_INPUT_FOLDER_PATH, input_filename+'_CKBCore_output.xlsx')
    ckb_output.to_excel(output_files_path, index=False)
    print(f"CKB Core Output is saved to {output_files_path}")
'''

'\nckb_output = getCKBCoreData(inp_df)\nif not ckb_output.empty:\n    output_files_path=os.path.join(USER_INPUT_FOLDER_PATH, input_filename+\'_CKBCore_output.xlsx\')\n    ckb_output.to_excel(output_files_path, index=False)\n    print(f"CKB Core Output is saved to {output_files_path}")\n'

In [27]:
#annotations runs SIFT indel, SIFT4G and dbNSFP processes
def annotations(inp_df, input_filename):

    print('SIFT Indel annotations have begun for ', input_filename)
    SIFT_indel(inp_df)
    print('SIFT 4G annotations have begun for',input_filename)
    SIFT4G_1(inp_df)
    print('dbNSFP annotations have begun for',input_filename)
    dbNSFP_1(inp_df)

    #Clinvar data Extraction
    Clinvar_output=clinvar_extraction(inp_df, "38")
    if not Clinvar_output.empty:
        output_files_path=os.path.join(USER_INPUT_FOLDER_PATH, input_filename+'_clinvar_output.xlsx')
        Clinvar_output.to_excel(output_files_path, index=False)
        print(f"Clinvar Output is saved to {output_files_path}")

    #CADD Output
    cadd_data = CADD_SNV(inp_df)
    if cadd_data:
        df_cadd = pd.DataFrame(cadd_data)
        # Reordering columns to have 'Chrom', 'Position', 'Ref', and 'Alt' at the beginning
        df_cadd = df_cadd[['Chrom', 'Position', 'Ref', 'Alt'] + [col for col in df_cadd.columns if col not in ['Chrom', 'Position', 'Ref', 'Alt']]]
        output_files_path = os.path.join(USER_INPUT_FOLDER_PATH, input_filename+'_cadd_data.xlsx')
        df_cadd.to_excel(output_files_path, index=False)
        print(f"Cadd Output is saved to {output_files_path}")
    
    #OncoKB processing
    oncokb_output = oncokb_open_page(inp_df, '33ddcd81-d4ed-4ede-9f24-93d1dacd65df')
    if not oncokb_output.empty:
        output_files_path=os.path.join(USER_INPUT_FOLDER_PATH, input_filename+'_OncoKB_output.xlsx')
        oncokb_output.to_excel(output_files_path, index=False)
        print(f"OncoKB Output is saved to {output_files_path}")
    
    #CKB Core 
    ckb_output = getCKBCoreData(inp_df)
    if not ckb_output.empty:
        output_files_path=os.path.join(USER_INPUT_FOLDER_PATH, input_filename+'_CKBCore_output.xlsx')
        ckb_output.to_excel(output_files_path, index=False)
        print(f"CKB Core Output is saved to {output_files_path}")

    #dbSNP
    dbsnp_output = dbsnp_extraction(inp_df,"38")
    if not dbsnp_output.empty:
        output_files_path=os.path.join(USER_INPUT_FOLDER_PATH, input_filename+'_dbSNP_output.xlsx')
        dbsnp_output.to_excel(output_files_path, index=False)
        print(f"dbSNP Output is saved to {output_files_path}")
    

#staus checks whether or not the file annotations is complete
def status(input_filename):
    SIFT_indel_annotation_file=os.path.exists(USER_INPUT_FOLDER_PATH+'SIFT_indel_'+input_filename+'.tsv')
    SIFT4G_annotation_file=os.path.exists(USER_INPUT_FOLDER_PATH+input_filename+'_SIFTannotations.xls')
    dbNSFP_annotation_file=os.path.exists(USER_INPUT_FOLDER_PATH+input_filename+'.vcf_dbNSFP.tsv')
    df_cadd_annotation_file=os.path.exists(USER_INPUT_FOLDER_PATH+input_filename+'_cadd_data.xlsx')
    Clinvar_output_annotation_file=os.path.exists(USER_INPUT_FOLDER_PATH+input_filename+'_clinvar_output.xlsx')
    oncokb_output_annotation_file=os.path.exists(USER_INPUT_FOLDER_PATH+input_filename+'_OncoKB_output.xlsx')
    ckb_output_annotation_file=os.path.exists(USER_INPUT_FOLDER_PATH+input_filename+'_CKBCore_output.xlsx')
    dbSNP_output_annotation_file=os.path.exists(USER_INPUT_FOLDER_PATH+input_filename+'_dbSNP_output.xlsx')
    
    display_refresh = False
    if SIFT_indel_annotation_file==True and SIFT4G_annotation_file ==True and dbNSFP_annotation_file==True and df_cadd_annotation_file==True and Clinvar_output_annotation_file==True and oncokb_output_annotation_file==True and ckb_output_annotation_file ==True and dbSNP_output_annotation_file== True:
        print('Annotations are complete')
    elif SIFT_indel_annotation_file==False:
        print('SIFT, SIFT 4G, dbNSFP, OncoKB, CADD and ClinVar annotations are going on')
        print('Please come back after sometime <Need to mention approx time based on input variants count>')
        display_refresh=True
    elif SIFT_indel_annotation_file==True and SIFT4G_annotation_file ==False and dbNSFP_annotation_file==False and df_cadd_annotation_file==False and Clinvar_output_annotation_file==False and oncokb_output_annotation_file==False:
        print('SIFT annotations are completed , SIFT4G, OncoKB, CADD and ClinVar and dbNSFP annotations are going on')
        display_refresh=True
    elif SIFT_indel_annotation_file==True and SIFT4G_annotation_file ==False and dbNSFP_annotation_file==False and df_cadd_annotation_file==False and Clinvar_output_annotation_file==True and oncokb_output_annotation_file==False:
        print('SIFT and ClinVar annotations are completed , SIFT4G, OncoKB, CADD and dbNSFP annotations are going on')
        display_refresh=True
    elif SIFT_indel_annotation_file==True and SIFT4G_annotation_file ==False and dbNSFP_annotation_file==False and df_cadd_annotation_file==True and Clinvar_output_annotation_file==True and oncokb_output_annotation_file==False:
        print('SIFT, CADD and ClinVar annotations are completed , SIFT4G, OncoKB and dbNSFP annotations are going on')
        display_refresh=True
    elif SIFT_indel_annotation_file==True and SIFT4G_annotation_file ==False and dbNSFP_annotation_file==False and df_cadd_annotation_file==True and Clinvar_output_annotation_file==True and oncokb_output_annotation_file==True:
        print('SIFT, CADD, OncoKB and ClinVar annotations are completed , SIFT4G and dbNSFP annotations are going on')
        display_refresh=True
    elif SIFT_indel_annotation_file==True and SIFT4G_annotation_file ==True and dbNSFP_annotation_file==False and df_cadd_annotation_file==True and Clinvar_output_annotation_file==True:
        print('SIFT, SIFT 4G, CADD and ClinVar annotations are completed , dbNSFP annotations is going on')
        display_refresh=True
    
    if display_refresh==True:
        previewButton1 = widgets.Button(description = "Refresh")
        previewButton1.on_click(functools.partial(click_refresh, input_filename=input_filename))
        display(previewButton1)
        
def click_refresh(b,input_filename):
    status(input_filename)

annotations(inp_df, input_filename)
status(input_filename)

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,HCC1187BL,HCC1187C,Gene name_BED file,Depth,Allele Frequency (%),REF_dbsnp ref,ALT_dbsnp alt,vtype,dbSNP_Id,gnomAD-Genomes
0,1,9665241,.,A,G,.,PASS,DP=130;MQ=185.80;FractionInformativeReads=0.977,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:47,0:0.000:24,0:23,0:47:.:.","0/1:71.13:54,26:0.325:32,11:22,15:80:28,26,18,...",PIK3CD,130,0.0,A,G,SNV,,
1,1,11216105,.,T,A,.,PASS,DP=155;MQ=198.79;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:-0.00:57,0:0.000:30,0:27,0:57:.:.","0/1:205.89:0,98:1.000:0,47:0,51:98:0,0,43,55:0...",MTOR,155,0.0,T,A,SNV,,
2,1,64931096,.,A,T,.,PASS,DP=206;MQ=105.45;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:50,0:0.000:25,0:25,0:50:.:.","0/1:126.57:81,75:0.481:33,36:48,39:156:46,35,3...",JAK1,206,0.0,A,T,SNV,,
3,1,64969201,.,G,GT,.,PASS,DP=210;MQ=214.69;FractionInformativeReads=0.886,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:45,0:0.000:18,0:27,0:45:.:.","0/1:110.62:74,67:0.475:34,37:40,30:141:35,39,2...",JAK1,210,0.0,,T,ins,,
4,1,65022230,.,G,A,.,PASS,DP=243;MQ=214.83;FractionInformativeReads=0.996,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:-0.00:62,0:0.000:31,0:31,0:62:.:.","0/1:132.48:105,75:0.417:56,39:49,36:180:45,60,...",JAK1,243,0.0,G,A,SNV,,
5,1,97142697,.,T,G,.,PASS,DP=116;MQ=190.00;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:-0.00:49,0:0.000:21,0:28,0:49:.:.","0/1:181.80:0,67:1.000:0,36:0,31:67:0,0,38,29:0...",DPYD,116,0.0,T,G,SNV,,
6,1,97177779,.,T,A,.,PASS,DP=116;MQ=195.08;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:40,0:0.000:19,0:21,0:40:.:.","0/1:92.37:41,35:0.461:18,14:23,21:76:13,28,14,...",DPYD,116,0.0,T,A,SNV,,
7,1,97371462,.,G,A,.,PASS,DP=109;MQ=190.06;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:46,0:0.000:20,0:26,0:46:.:.","0/1:102.48:33,30:0.476:18,9:15,21:63:15,18,19,...",DPYD,109,0.0,G,A,SNV,1352251059,0.000007
8,1,97527794,.,GA,G,.,PASS,DP=132;MQ=193.39;FractionInformativeReads=0.780,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:32,2:0.059:17,1:15,1:34:.:.","0/1:56.78:31,38:0.551:14,20:17,18:69:15,16,15,...",DPYD,132,5.9,A,,del,,
9,1,156865995,.,A,G,.,PASS,DP=238;MQ=208.22;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:72,0:0.000:38,0:34,0:72:.:.","0/1:27.68:158,8:0.048:73,2:85,6:166:88,70,3,5:...",NTRK1,238,0.0,A,G,SNV,,


no:1,pos:9665241,ref:A,alt:G
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=snp&term=((1[Chromosome])%20AND%209665241[Base%20Position])%20AND%20SNV
Error occured 'NoneType' object is not subscriptable
no:1,pos:11216105,ref:T,alt:A
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=snp&term=((1[Chromosome])%20AND%2011216105[Base%20Position])%20AND%20SNV
Error occured 'NoneType' object is not subscriptable
no:1,pos:64931096,ref:A,alt:T
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=snp&term=((1[Chromosome])%20AND%2064931096[Base%20Position])%20AND%20SNV
Error occured 'NoneType' object is not subscriptable
no:1,pos:64969201,ref:,alt:T
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=snp&term=((1[Chromosome])%20AND%2064969201[Base%20Position])%20AND%20ins
Error occured 'NoneType' object is not subscriptable
no:1,pos:65022230,ref:G,alt:A
https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=snp&term=((1[Chromosome])%20AND%2065022

TO CREATE A DISPLAY TABLE

In [32]:
def displayVariants(file):
    file_name = os.path.splitext(file)[0]
    
    input_vcf_df=read_vcf(USER_INPUT_FOLDER_PATH+file)
    input_vcf_df["snp_pos"] = input_vcf_df["#CHROM"].astype(str) + ","+input_vcf_df["POS"].astype(str)
    chr_no=list(input_vcf_df['#CHROM'])
    print('Total Number of Variants:',len(chr_no))
        
    display(input_vcf_df.head(2))
    
    #Integration of SIFT INDEL 
    try:
        #trying to open a file in read mode
        df_SIFT = pd.read_csv(USER_INPUT_FOLDER_PATH+'SIFT_indel_'+file_name+'.tsv',sep='\t') 
        print('SIFT Indel OUTPUT')
        display(df_SIFT.head(2))
    except FileNotFoundError:
        print("SIFT Annotations in progress..")
        return
    except:
        print("Other error")
        return
    # Splitting 'Coordinates' column into separate columns
    df_SIFT[['N1', 'N2', 'UK1','UK2','UK3']] = df_SIFT['Coordinates'].str.split(',', expand=True)
    #display(df_SIFT.head(2))
    # Combining 'N1' and 'N2' columns to create 'snp_pos' column
    df_SIFT['snp_pos']=df_SIFT['N1'].astype(str) + ','+df_SIFT['N2'].astype(str)
    
    # Selecting required columns from SIFT INDEL 
    df_SIFT=df_SIFT[['snp_pos','Gene Name','Amino acid position change','Amino acid change','Substitution Type','Effect','Region']]
    #print('DF--------------')
    #display(df.head(2))
    # Merging SIFT dataframe with input_vcf_df on 'snp_pos' column
    merged = pd.merge(input_vcf_df,df_SIFT, how='left', on='snp_pos')
    # Renaming columns according to the display table requirement
    merged.rename(columns = {'Amino acid position change':'Amino acid position','Amino acid change_y':'Amino acid change','Substitution Type':'Mutation type','Effect':'SIFT4G prediction'}, inplace = True)
    print('Merged SIFT Indel-------------')
    display(merged.head(2))
    

    #Intergration of SIFT 4G
    try:
        sift_4g_file=TEMP_FOLDER_PATH+file_name+'_SIFTannotations.xls'
    except FileNotFoundError:
        print("SIFT4G Annotations in progress..")
        return
    except:
        print("Other error")
        return

    sift_4g_df = pd.read_csv(sift_4g_file, sep='\t')
    print('SIFT 4G output')
    display(sift_4g_df.head(2))
    # Combining chromosome and position to create 'snp_pos' column
    sift_4g_df["snp_pos"] = sift_4g_df["CHROM"].astype(str) + "," + sift_4g_df["POS"].astype(str)
    sift_4g_df['snp_pos'] = sift_4g_df['snp_pos'].str.replace('chr', '') 
    sift_4g_df['aa_change'] = sift_4g_df["REF_AMINO"].astype(str) + sift_4g_df["AMINO_POS"].astype(str) + sift_4g_df["ALT_AMINO"].astype(str)
    
    # Merging SIFT 4G dataframe with the previously merged dataframe on 'snp_pos' column
    merged_SIFT4G = pd.merge(merged, sift_4g_df, how='left', on='snp_pos')
    merged_SIFT4G.replace(np.NaN, '', inplace=True)
    merged_SIFT4G.replace(np.nan, '', inplace=True)
    merged_SIFT4G.replace(to_replace=["nan", 'nannannan'], value="", inplace=True)
    
    # Concatenating relevant columns from SIFT INDEL AND SIFT 4G 
    merged_SIFT4G["Gene_Name"] = merged_SIFT4G["Gene Name"] + merged_SIFT4G["GENE_NAME"]
    merged_SIFT4G["Amino_acid_position"] = merged_SIFT4G["Amino acid position"] + merged_SIFT4G["AMINO_POS"].astype(str)
    merged_SIFT4G["Amino_acid_change"] = merged_SIFT4G["Amino acid change"] + merged_SIFT4G["aa_change"]
    merged_SIFT4G["Mutation_type"] = merged_SIFT4G["Mutation type"] + merged_SIFT4G["VARIANT_TYPE"]
    merged_SIFT4G["SIFT4G_prediction"] = merged_SIFT4G["SIFT4G prediction"] + merged_SIFT4G["SIFT_PREDICTION"]
    
    merged_SIFT4G['HGVS nomenclature'] = ''
    merged_SIFT4G['Amino_acid_position'] = merged_SIFT4G['Amino_acid_position'].astype(str).apply(lambda x: x.replace('.0', ''))
    merged_SIFT4G['Amino_acid_change'] = merged_SIFT4G['Amino_acid_change'].astype(str).apply(lambda x: x.replace('.0', ''))
    merged_SIFT4G.rename(columns={'dbSNP':'dbsnp', 'POS_x': 'POS'}, inplace=True)
    merged_SIFT4G['dbsnp'] = merged_SIFT4G['dbsnp'].apply(lambda x: x if isinstance(x, str) and (x.startswith('rs') or x == 'novel') else '')
    merged_SIFT4G['REGION'] = merged_SIFT4G['Region'] + merged_SIFT4G['REGION']
    
    # Dropping unnecessary columns
    columns_to_drop = ['Region', 'CHROM', 'POS_y', 'REF_ALLELE', 'ALT_ALLELE', 'REF_AMINO', 'ALT_AMINO', 
                       'Gene Name', 'Mutation type', 'Amino acid change', 'SIFT4G prediction', 'Amino acid position', 
                       'GENE_NAME', 'VARIANT_TYPE', 'SIFT_PREDICTION', 'AMINO_POS', 'aa_change', 'TRANSCRIPT_ID', 
                       'GENE_ID', 'SIFT_SCORE', 'SIFT_MEDIAN', 'NUM_SEQS']
    merged_SIFT4G.drop(columns=columns_to_drop, axis=1, inplace=True)
    
    print("merged_SIFT4G")
    display(merged_SIFT4G.head(2))
    
    #Integration of dbNSFP
    try:
        #db_path=file_path+file_new+'.vcf.tsv'
        dbnsfp_path=USER_INPUT_FOLDER_PATH+file_name+'.vcf_dbNSFP.tsv'
    except FileNotFoundError:
        print("dbNSFP Annotations in progress..")
        return
    except:
        print("Other error")
        return

    # Reading dbNSFP file
    dbnsfp_df = pd.read_csv(dbnsfp_path, sep='\t', encoding='latin1')
    print('dbNSFP output')
    display(dbnsfp_df.head(2))
    dbnsfp_df['dbNSFP Amino acid change'] = dbnsfp_df['MutationTaster_AAE'].astype(str).str.replace(".;", "").str.replace(";", " ")
    dbnsfp_df['POS']=list(dbnsfp_df['POS'])
    
    #The check_splicesite function determines if a row in the DataFrame corresponds to a splicesite mutation by checking if both 'aaref' and 'aaalt' are either '.' or non-string/empty values, and labels such rows as 'Splicesite'. This function is then applied to each row in dbnsfp_df, and the result is stored in the new column 'Mutation_Type'.
    def check_splicesite(row):
        if (row['aaref'] == '.' and row['aaalt'] == '.' or
            not isinstance(row['aaref'], str) and not isinstance(row['aaalt'], str) or
            row['aaref'] == '' and row['aaalt'] == ''):
            return 'Splicesite'
        return '' 
    dbnsfp_df['Mutation_Type'] = dbnsfp_df.apply(check_splicesite, axis=1)
    
    # Selecting required columns from dbNSFP dataframe
    dbnsfp_df['aapos']=dbnsfp_df['aapos'].str.split(';').apply(set).str.join(',')
    dbnsfp_df['genename']=dbnsfp_df['genename'].str.split(';').apply(set).str.join(',')
    dbnsfp_df=dbnsfp_df[['POS','aapos','genename','MutationTaster_AAE','rs_dbSNP','Mutation_Type', 'dbNSFP Amino acid change']]
    dbnsfp_df['rs_dbSNP'] = dbnsfp_df['rs_dbSNP'].apply(lambda x: x if isinstance(x, str) and (x.startswith('rs') or x == 'novel') else '')
    
    # Merging dbNSFP dataframe with merged_SIFT4G dataframe on 'POS' column
    all_columns = list(dbnsfp_df) 
    dbnsfp_df[all_columns] = dbnsfp_df[all_columns].astype(str)
    all_columns = list(merged_SIFT4G) 
    merged_SIFT4G[all_columns] = merged_SIFT4G[all_columns].astype(str)
    merged_SIFT4G_dbNSFP = pd.merge(merged_SIFT4G,dbnsfp_df, how='left', on='POS')

    # Concatenating relevant columns between dbNSFP dataframe and merged_SIFT4G dataframe
    merged_SIFT4G_dbNSFP["dbSNP"] = merged_SIFT4G_dbNSFP["dbsnp"] +' '+merged_SIFT4G_dbNSFP["rs_dbSNP"]
    merged_SIFT4G_dbNSFP["Amino acid position"] = merged_SIFT4G_dbNSFP["Amino_acid_position"]+' '+merged_SIFT4G_dbNSFP["aapos"]
    merged_SIFT4G_dbNSFP["Gene Name"] = merged_SIFT4G_dbNSFP["Gene_Name"]+' '+merged_SIFT4G_dbNSFP["genename"]
    merged_SIFT4G_dbNSFP["Amino acid change"] = merged_SIFT4G_dbNSFP["Amino_acid_change"]
    merged_SIFT4G_dbNSFP["Mutation type"] = merged_SIFT4G_dbNSFP["Mutation_type"]+' '+merged_SIFT4G_dbNSFP["Mutation_Type"]

    # Additional modifications to clean the column 
    merged_SIFT4G_dbNSFP['dbSNP'] = merged_SIFT4G_dbNSFP['dbSNP'].astype(str).replace('nan', '')
    merged_SIFT4G_dbNSFP['dbSNP'] = merged_SIFT4G_dbNSFP['dbSNP'].str.split(' ').apply(set).str.join(' ')
    merged_SIFT4G_dbNSFP['dbNSFP Amino acid change'] = merged_SIFT4G_dbNSFP['dbNSFP Amino acid change'].astype(str).replace('nan', '')
    merged_SIFT4G_dbNSFP['dbNSFP Amino acid change'] = merged_SIFT4G_dbNSFP['dbNSFP Amino acid change'].str.split(' ').apply(set).str.join(' ')
    merged_SIFT4G_dbNSFP['dbNSFP Amino acid change'] = merged_SIFT4G_dbNSFP['dbNSFP Amino acid change'].str.replace('.', '')
    merged_SIFT4G_dbNSFP['Gene Name'] = merged_SIFT4G_dbNSFP['Gene Name'].astype(str)
    merged_SIFT4G_dbNSFP['Gene Name']=merged_SIFT4G_dbNSFP['Gene Name'].str.split(' ').apply(set).str.join(' ')
    merged_SIFT4G_dbNSFP['Gene Name'] = merged_SIFT4G_dbNSFP['Gene Name'].str.replace(' ', '')
    merged_SIFT4G_dbNSFP['Mutation type']=merged_SIFT4G_dbNSFP['Mutation type'].astype(str)
    merged_SIFT4G_dbNSFP['Mutation type']=merged_SIFT4G_dbNSFP['Mutation type'].str.split(' ').apply(set).str.join(' ')
    
    # Dropping unnecessary columns
    merged_SIFT4G_dbNSFP.drop(['dbsnp','rs_dbSNP','Gene_Name','Mutation_Type','Amino_acid_change','Amino_acid_position','genename','Mutation_type','aapos',"MutationTaster_AAE"], axis =1, inplace=True)
    # Reordering columns
    merged_SIFT4G_dbNSFP=merged_SIFT4G_dbNSFP[['#CHROM','POS','REF','ALT','FILTER','Depth', 'Allele Frequency (%)', 'Gene name_BED file','snp_pos','SIFT4G_prediction','dbSNP','Amino acid position','Gene Name','Amino acid change','dbNSFP Amino acid change','REGION','Mutation type','HGVS nomenclature']]
    
    # Handling missing values in 'Gene Name' column
    # Filtering rows with missing 'Gene Name' in merged_SIFT4G_dbNSFP dataframe
    db_fill_gene=merged_SIFT4G_dbNSFP.loc[merged_SIFT4G_dbNSFP['Gene Name'] == '']
    # Reading dbNSFP file again to fill missing values
    dbnsfp_fill = pd.read_csv(dbnsfp_path, sep='\t', encoding='latin1')

    # Creating a dataframe to store the relevant columns from dbNSFP for filling missing values
    db_fill_gene1=pd.DataFrame()
    db_fill_gene1['#CHROM']=list(dbnsfp_fill['#CHROM'])
    db_fill_gene1['POS']=list(dbnsfp_fill['POS'])
    db_fill_gene1['Amino acid position']=list(dbnsfp_fill['aapos'])
    db_fill_gene1['Gene Name']=list(dbnsfp_fill['genename'])
    db_fill_gene1['dbSNP']=list(dbnsfp_fill['rs_dbSNP'])
    
    db_fill_gene1['SIFT4G_prediction']=list(dbnsfp_fill['SIFT4G_pred'])
    
    db_fill_gene1['Gene Name'] = db_fill_gene1['Gene Name'].str.split(';').str[0]
    # Replacing 'D' with 'DELETERIOUS' in 'SIFT4G_prediction' column
    db_fill_gene1['SIFT4G_prediction'].replace({'D':'DELETERIOUS'}, regex=True, inplace=True)

    # Merging db_fill_gene and db_fill_gene1 dataframes to fill missing values in merged_SIFT4G_dbNSFP dataframe
    db_fill_gene['#CHROM']=db_fill_gene['#CHROM'].astype(str)
    db_fill_gene1['#CHROM']=db_fill_gene1['#CHROM'].astype(str)
    db_fill_gene1['POS']=db_fill_gene1['POS'].astype(str)
    db_fil = pd.merge(db_fill_gene,db_fill_gene1, how='left', on=['#CHROM','POS'])
    
    # Dropping unnecessary columns and renaming columns
    db_fil.drop(['SIFT4G_prediction_x', 'dbSNP_x', 'Amino acid position_x','Gene Name_x'], axis = 1, inplace=True)
    db_fil.rename(columns = {'Amino acid position_y':'Amino acid position', 'Gene Name_y':'Gene Name', 'dbSNP_y':'dbSNP','SIFT4G_prediction_y':'SIFT4G_prediction'}, inplace = True)
    # Reordering columns
    db_fil=db_fil[['#CHROM', 'POS', 'REF', 'ALT', 'FILTER', 'Depth', 
           'snp_pos', 'SIFT4G_prediction','dbSNP','Amino acid position', 'Gene Name','Amino acid change',
           'dbNSFP Amino acid change', 'REGION', 'Mutation type',
           'HGVS nomenclature']]
    
    # Setting values for 'REGION' and 'Mutation type' columns where 'Gene Name' is not missing
    db_fil.loc[db_fil['Gene Name'].isnull() == False, 'REGION'] = 'CDS'                       
    db_fil.loc[db_fil['Gene Name'].isnull() == False, 'Mutation type'] = 'NONSYNONYMOUS'  
    
    # Concatenating the filled dataframe with the original dataframe
    merged_SIFT4G_dbNSFP1=merged_SIFT4G_dbNSFP.copy(deep=True)
    merged_SIFT4G_dbNSFP1.drop(merged_SIFT4G_dbNSFP1.index[merged_SIFT4G_dbNSFP1['Gene Name'] == ''], inplace=True)
    merged_SIFT4G_dbNSFP2 = pd.concat([merged_SIFT4G_dbNSFP1, db_fil])
    
    # Sorting the concatenated dataframe
    def custom_sort_key(value):
        if pd.isna(value): # If the value is NaN, assign it a priority of 1
            return (1, value)  
        elif isinstance(value, str) and value.isdigit(): # If the value is a string of digits, convert it to int and assign a priority of 2
            return (2, int(value))  
        else: # For all other cases, assign a priority of 3
            return (3, value) 
            
    # Apply the custom sorting key function to the '#CHROM' column of the merged dataframe
    merged_SIFT4G_dbNSFP2['_sort_key'] = merged_SIFT4G_dbNSFP2['#CHROM'].apply(custom_sort_key)
    # Sort the dataframe based on the '_sort_key' column
    merged_SIFT4G_dbNSFP2.sort_values(by='_sort_key', inplace=True)
    merged_SIFT4G_dbNSFP2.drop('_sort_key', axis=1, inplace=True)
    merged_SIFT4G_dbNSFP2.reset_index(drop=True, inplace=True)
    merged_SIFT4G_dbNSFP3 = merged_SIFT4G_dbNSFP2.copy(deep=True)

    # Creating 'snp_pos1' column
    merged_SIFT4G_dbNSFP3["snp_pos1"] = 'chr'+merged_SIFT4G_dbNSFP3["#CHROM"].astype(str) +':'+ merged_SIFT4G_dbNSFP3["POS"].astype(str)
    merged_SIFT4G_dbNSFP3.snp_pos1 = merged_SIFT4G_dbNSFP3.snp_pos1.str.replace(' ', '')
    merged_SIFT4G_dbNSFP3["snp_pos1"]
    
    print('merged_dbNSFP3')
    display(merged_SIFT4G_dbNSFP3.head(2))

    #Integration of CADD-SNV
    try:
        cadd_path = USER_INPUT_FOLDER_PATH+file_name+'_cadd_data.xlsx'
    except FileNotFoundError:
        print("CADD Annotations in progress..")
        return
    except:
        print("Other error")
        return
    cadd_df = pd.read_excel(cadd_path)
    print('CADD output')
    display(cadd_df.head(2))
    cadd_df['snp_pos1'] = 'chr'+cadd_df['Chrom'].astype(str) +':'+ cadd_df['Position'].astype(str)
    cadd_selected_columns = ['Consequence', 'Exon', 'Intron', 'PHRED', 'Grantham', 'snp_pos1']
    cadd_df = cadd_df[cadd_selected_columns]
    new_column_names = {'Consequence': 'CADD_Consequence', 'Exon': 'Exon Number', 'Intron': 'Intron number', 'PHRED': 'CADD_PHRED', 'Grantham': 'CADD_Grantham'}
    cadd_df = cadd_df.rename(columns=new_column_names)
    def determine_cadd_prediction(phred_value):
        if pd.isnull(phred_value) or phred_value == '':
            return ''  
        elif phred_value >= 15:
            return 'Deleterious'
        else:
            return 'Benign'
    cadd_df['CADD_Prediction'] = cadd_df['CADD_PHRED'].apply(determine_cadd_prediction)
    merged_dbNSFP1_cadd= pd.merge(merged_SIFT4G_dbNSFP3, cadd_df, how='left', on='snp_pos1')
    merged_dbNSFP1_cadd = merged_dbNSFP1_cadd.drop_duplicates()
    print('merged_CADD')
    display(merged_dbNSFP1_cadd.head(2))
    
    #Integration of ClinVar
    try:
        clinvar_path = USER_INPUT_FOLDER_PATH+file_name+'_clinvar_output.xlsx'
    except FileNotFoundError:
        print("ClinVar Annotations in progress..")
        return
    except:
        print("Other error")
        return
    clinvar_file = pd.read_excel(clinvar_path)
    print('Clinvar output')
    display(clinvar_file.head(2))
    clinvar_file['snp_pos1'] = 'chr'+clinvar_file["#CHROM"].astype(str) +':'+ clinvar_file["POS"].astype(str)
    clinvar_file.drop(['#CHROM', 'POS', 'REF', 'ALT', 'FILTER', 'Depth', 'Allele Frequency (%)'], axis=1, inplace=True)
    merged_cadd_clinvar= pd.merge(merged_dbNSFP1_cadd,clinvar_file, how='left', on='snp_pos1')
    merged_cadd_clinvar.drop(['Gene name_BED file_y'], axis = 1, inplace=True)
    merged_cadd_clinvar.rename(columns = {'Gene name_BED file_x':'Gene name_BED file'}, inplace = True)
    desired_column_order = ['#CHROM', 'POS', 'REF', 'ALT', 'FILTER', 'Depth', 'Allele Frequency (%)', 'Gene name_BED file','SIFT4G_prediction', 'dbSNP', 'Amino acid position', 'Gene Name', 'Amino acid change', 'dbNSFP Amino acid change', 'REGION', 'Mutation type',  'HGVS nomenclature', 'Gene', 'Clinvar_accession_no', 'Clinvar_Protein_change', 'gnomAD_value', '1000_Genomes_value', 'ExAC_value', 'dbSNP_id', 'OMIM_id', 'Germline_classification', 'Somatic_clinical_impact', 'Clinvar_link', 'CADD_Consequence', 'Exon Number', 'Intron number', 'CADD_PHRED', 'CADD_Prediction', 'CADD_Grantham', 'snp_pos1']
    merged_cadd_clinvar = merged_cadd_clinvar[desired_column_order]
    print('merged_clinvar')
    display(merged_cadd_clinvar.head(2))
    
    #Integration of Cancer hotspot
    merged_clinvar_cancerhotspot = cancer_hotspot_annotation_for_whole_file(merged_cadd_clinvar)
    
    #Integration of Intogen
    merged_cancerhotspot_intogen_out=intogen_annotation(merged_clinvar_cancerhotspot)
    
       
    #Integration of OncoKB 
    try:
        oncokb_path = USER_INPUT_FOLDER_PATH+file_name+'_OncoKB_output.xlsx'
    except FileNotFoundError:
        print("OncoKB Annotations in progress..")
        return
    except:
        print("Other error")
        return
    oncokb_file = pd.read_excel(oncokb_path)
    print('OncoKb Output')
    display(oncokb_file.head(2))
    oncokb_file['snp_pos1'] = 'chr'+oncokb_file["chr_no"].astype(str) +':'+ oncokb_file["chr_pos"].astype(str)
    oncokb_file['OncoKB_link']="https://www.oncokb.org/hgvsg/"+oncokb_file["chr_no"].astype(str)+":g."+oncokb_file["chr_pos"].astype(str)+oncokb_file["ref"].astype(str)+'>'+oncokb_file["alt"].astype(str)+"?refGenome=GRCh38"
    oncokb_file.rename(columns = {'oncogenic':'OncoKB_oncogenic_status', 'mutation_effect_known_effect':'OncoKB_mutation_effect'}, inplace=True)
    desired_column_order = ['OncoKB_link', 'OncoKB_oncogenic_status', 'OncoKB_mutation_effect', 'snp_pos1']
    oncokb_file=oncokb_file[desired_column_order]
    oncokb_file['OncoKB_oncogenic_status'] = oncokb_file['OncoKB_oncogenic_status'].replace('Unknown', '')
    oncokb_file['OncoKB_mutation_effect'] = oncokb_file['OncoKB_mutation_effect'].replace('Unknown', '')
    merged_intogen_oncokb= pd.merge(merged_cancerhotspot_intogen_out,oncokb_file, how='left', on='snp_pos1')

    #Integrating CKB Core
    try:
        ckb_path = USER_INPUT_FOLDER_PATH+file_name+'_CKBCore_output.xlsx'
    except FileNotFoundError:
        print("CKB Core Annotations in progress..")
        return
    except:
        print("Other error")
        return
    ckb_df = pd.read_excel(ckb_path)
    print('CKb output')
    display(ckb_df.head(2))
    ckb_df['snp_pos1'] = 'chr'+ckb_df["#CHROM"].astype(str) +':'+ ckb_df["POS"].astype(str)
    ckb_df.drop(['#CHROM', 'POS', 'REF', 'ALT', 'FILTER', 'ID', 'QUAL', 'INFO', 'FORMAT', 'HCC1187BL', 'HCC1187C', 'Gene name_BED file', 'Depth', 'Allele Frequency (%)', 'Gene', 'Clinvar_accession_no', 'Clinvar_Protein_change', 'gnomAD_value', '1000_Genomes_value', 'ExAC_value', 'dbSNP_id', 'OMIM_id', 'Germline_classification', 'Somatic_clinical_impact', 'Clinvar_link'], axis=1, inplace=True)
    merged_ckb= pd.merge(merged_intogen_oncokb,ckb_df, how='left', on='snp_pos1')
    # merged_ckb.drop(['snp_pos1'], axis =1, inplace=True)
    desired_order = ['CHROM', 'POS', 'REF', 'ALT', 'FILTER', 'Depth', 'Allele Frequency (%)', 'Gene name_BED file','SIFT4G_prediction', 'dbSNP', 'Amino acid position',  'Gene Name', 'Amino acid change', 'dbNSFP Amino acid change', 'REGION',  'Mutation type', 'HGVS nomenclature', 'Gene', 'Clinvar_accession_no', 'Clinvar_Protein_change', 'gnomAD_value', '1000_Genomes_value', 'ExAC_value', 'dbSNP_id', 'OMIM_id', 'Germline_classification', 'Somatic_clinical_impact',  'Clinvar_link', 'CADD_Consequence', 'Exon Number', 'Intron number', 'CADD_PHRED', 'CADD_Prediction', 'CADD_Grantham', 'Cancer Hotspot_Gene mutation_Sample frequency', 'Cancer Hotspot_Gene mutation_Sample frequency-Tumor_type_composition', 'Cancer Hotspot_Mutation site_Sample frequency', 'Cancer Hotspot_Mutation site_Sample frequency-Tumor_type_composition', 'Cancer Hotspot_Exact Mutation_Sample frequency', 'Intogen_Gene Mutation_Sample frequency', 'Intogen_Mutation Site_Sample frequency', 'Intogen_Exact Mutation_Sample frequency', 'Intogen_Known Driver', 'OncoKB_link', 'OncoKB_oncogenic_status', 'OncoKB_mutation_effect', 'NCBI Gene ID', 'CKB Core _Synonyms', 'CKB Core_GeneRole', 'CKB Core_Gene_Description']
    print('merged_ckb')
    display(merged_ckb.head(2))

    #integrating dbSNP
    try:
        dbsnp_path = USER_INPUT_FOLDER_PATH+file_name+'_dbSNP_output.xlsx'
    except FileNotFoundError:
        print("dbSNP Annotations in progress..")
        return
    except:
        print("Other error")
        return
    dbsnp_df = pd.read_excel(dbsnp_path)
    print('dbsnp output')
    display(dbsnp_df.head(2))
    dbsnp_df['snp_pos1'] = 'chr'+dbsnp_df["#CHROM"].astype(str) +':'+ dbsnp_df["POS"].astype(str)
    dbsnp_df.drop(['#CHROM', 'POS', 'REF', 'ALT', 'FILTER', 'ID', 'QUAL', 'INFO', 'FORMAT', 'HCC1187BL', 'HCC1187C', 'Gene name_BED file', 'Depth', 'Allele Frequency (%)'], axis=1, inplace=True)
    merged_dbsnp= pd.merge(merged_ckb,dbsnp_df, how='left', on='snp_pos1')
    merged_dbsnp.drop(['snp_pos1'], axis =1, inplace=True)
    desired_order = ['#CHROM', 'POS', 'REF', 'ALT', 'FILTER', 'Depth', 'Allele Frequency (%)', 'Gene name_BED file','SIFT4G_prediction', 'dbSNP', 'dbSNP_rs_id','Amino acid position',  'Gene Name', 'Amino acid change', 'dbNSFP Amino acid change', 'REGION',  'Mutation type', 'HGVS nomenclature', 'Gene', 'Clinvar_accession_no', 'Clinvar_Protein_change', 'gnomAD_value',"gnomAD_value_dbSNP", '1000_Genomes_value', 'ExAC_value', 'dbSNP_id', 'OMIM_id', 'Germline_classification', 'Somatic_clinical_impact',  'Clinvar_link', 'CADD_Consequence', 'Exon Number', 'Intron number', 'CADD_PHRED', 'CADD_Prediction', 'CADD_Grantham', 'Cancer Hotspot_Gene mutation_Sample frequency', 'Cancer Hotspot_Gene mutation_Sample frequency-Tumor_type_composition', 'Cancer Hotspot_Mutation site_Sample frequency', 'Cancer Hotspot_Mutation site_Sample frequency-Tumor_type_composition', 'Cancer Hotspot_Exact Mutation_Sample frequency', 'Intogen_Gene Mutation_Sample frequency', 'Intogen_Mutation Site_Sample frequency', 'Intogen_Exact Mutation_Sample frequency', 'Intogen_Known Driver', 'OncoKB_link', 'OncoKB_oncogenic_status', 'OncoKB_mutation_effect', 'NCBI Gene ID', 'CKB Core _Synonyms', 'CKB Core_GeneRole', 'CKB Core_Gene_Description']
    merged_dbsnp=merged_dbsnp[desired_order]
    print("merged_dbsnp")
    display(merged_dbsnp.head(2))
    
    excel_file_path = USER_INPUT_FOLDER_PATH+input_filename+"_Display_Table.xlsx"
    merged_dbsnp.to_excel(excel_file_path, index=False)
    print(f"DataFrame saved to {excel_file_path}")
    display(merged_dbsnp.head(2))

#displayVariants('somatic_snv_and_indel_with_new_variants.vcf')
displayVariants(input_file_path)

Total Number of Variants: 176


,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,HCC1187BL,HCC1187C,Gene name_BED file,Depth,Allele Frequency (%),snp_pos
0,1,9665241,.,A,G,.,PASS,DP=130;MQ=185.80;FractionInformativeReads=0.977,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:47,0:0.000:24,0:23,0:47:.:.","0/1:71.13:54,26:0.325:32,11:22,15:80:28,26,18,...",PIK3CD,130,0.0,"1,9665241"
1,1,11216105,.,T,A,.,PASS,DP=155;MQ=198.79;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:-0.00:57,0:0.000:30,0:27,0:57:.:.","0/1:205.89:0,98:1.000:0,47:0,51:98:0,0,43,55:0...",MTOR,155,0.0,"1,11216105"


SIFT Indel OUTPUT


,Coordinates,Gene ID,Transcript ID,Substitution Type,Region,Amino acid position change,Effect,Confidence Score,Classification Path,Nucleotide change,Amino acid change,Indel Location,Causes NMD,Repeat detected,Transcript visualization,Gene Name,Gene Desc,Protein Family ID,Protein Family Desc,Transcript Status,Protein Family Size,Ka/Ks (Mouse),Ka/Ks (Macaque),OMIM Disease,1000 Genomes
0,"8,92007467,92007468,1,-/",ENSG00000079102,ENST00000265814,NaN,INTRON,NaN,NaN,NaN,NaN,TAATT-a-AAAAA,NaN,31%,NaN,NaN,<---{}[]--[]--[]--[]--[]--[]--[]-*.-[]--[]--[]...,RUNX1T1,runt-related transcription factor 1; transloca...,ENSFM00250000001209,CBF MTG8,KNOWN,42.0,0.103279,0.418142,"RUNT-RELATED TRANSCRIPTION FACTOR 1, TRANSLOCA...",NaN
1,"7,55053312,55053323,1,-/",ENSG00000146648,ENST00000275493,NaN,INTRON,NaN,NaN,NaN,NaN,GGAGG-gtgcatctcta-GAAAT,NaN,2%,NaN,NaN,|---{}[]-.*-[]--[]--[]--[]--[]--[]--[]--[]--[]...,EGFR,epidermal growth factor receptor [Source:HGNC ...,ENSFM00410000138465,RECEPTOR TYROSINE KINASE ERBB PRECURSOR EC_2.7...,KNOWN,36.0,0.068909,0.040964,EPIDERMAL GROWTH FACTOR RECEPTOR; EGFR;;V-ERB-...,NaN


Merged SIFT Indel-------------


...

,#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,HCC1187BL,HCC1187C,Gene name_BED file,Depth,Allele Frequency (%),dbSNP_Id,gnomAD-Genomes,dbSNP_rs_id,gnomAD_value_dbSNP
0,1,9665241,.,A,G,.,PASS,DP=130;MQ=185.80;FractionInformativeReads=0.977,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:0.00:47,0:0.000:24,0:23,0:47:.:.","0/1:71.13:54,26:0.325:32,11:22,15:80:28,26,18,...",PIK3CD,130,0.0,NaN,NaN,NaN,NaN
1,1,11216105,.,T,A,.,PASS,DP=155;MQ=198.79;FractionInformativeReads=1.000,GT:SQ:AD:AF:F1R2:F2R1:DP:SB:MB,"0/0:-0.00:57,0:0.000:30,0:27,0:57:.:.","0/1:205.89:0,98:1.000:0,47:0,51:98:0,0,43,55:0...",MTOR,155,0.0,NaN,NaN,NaN,NaN


merged_dbsnp


,#CHROM,POS,REF,ALT,FILTER,Depth,Allele Frequency (%),Gene name_BED file,SIFT4G_prediction,dbSNP,dbSNP_rs_id,Amino acid position,Gene Name,Amino acid change,dbNSFP Amino acid change,REGION,Mutation type,HGVS nomenclature,Gene,Clinvar_accession_no,Clinvar_Protein_change,gnomAD_value,gnomAD_value_dbSNP,1000_Genomes_value,ExAC_value,dbSNP_id,OMIM_id,Germline_classification,Somatic_clinical_impact,Clinvar_link,CADD_Consequence,Exon Number,Intron number,CADD_PHRED,CADD_Prediction,CADD_Grantham,Cancer Hotspot_Gene mutation_Sample frequency,Cancer Hotspot_Gene mutation_Sample frequency-Tumor_type_composition,Cancer Hotspot_Mutation site_Sample frequency,Cancer Hotspot_Mutation site_Sample frequency-Tumor_type_composition,Cancer Hotspot_Exact Mutation_Sample frequency,Intogen_Gene Mutation_Sample frequency,Intogen_Mutation Site_Sample frequency,Intogen_Exact Mutation_Sample frequency,Intogen_Known Driver,OncoKB_link,OncoKB_oncogenic_status,OncoKB_mutation_effect,NCBI Gene ID,CKB Core _Synonyms,CKB Core_GeneRole,CKB Core_Gene_Description
0,1,9665241,A,G,PASS,130,0.0,PIK3CD,,,NaN,NaN,nan,,,,nan,,NaN,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,INTRONIC,NaN,1/23,3.331,Benign,NaN,,,,,,None,None,None,None,https://www.oncokb.org/hgvsg/1:g.9665241A>G?re...,,,5293,APDS | IMD14 | IMD14A | IMD14B | p110D | P110D...,Unknown,"PIK3CD, phosphatidylinositol 4,5-bisphosphate ..."
1,1,9665241,A,G,PASS,130,0.0,PIK3CD,,,NaN,NaN,nan,,,,nan,,NaN,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UPSTREAM,NaN,NaN,3.331,Benign,NaN,,,,,,,None,None,,https://www.oncokb.org/hgvsg/1:g.9665241A>G?re...,,,5293,APDS | IMD14 | IMD14A | IMD14B | p110D | P110D...,Unknown,"PIK3CD, phosphatidylinositol 4,5-bisphosphate ..."


DataFrame saved to /home/ubuntu/jupyter_notebooks/Internal/input_files/Somatic_SNV_with_pathogenic_var.vcf_cgp_snv_indel_Display_Table.xlsx


,#CHROM,POS,REF,ALT,FILTER,Depth,Allele Frequency (%),Gene name_BED file,SIFT4G_prediction,dbSNP,dbSNP_rs_id,Amino acid position,Gene Name,Amino acid change,dbNSFP Amino acid change,REGION,Mutation type,HGVS nomenclature,Gene,Clinvar_accession_no,Clinvar_Protein_change,gnomAD_value,gnomAD_value_dbSNP,1000_Genomes_value,ExAC_value,dbSNP_id,OMIM_id,Germline_classification,Somatic_clinical_impact,Clinvar_link,CADD_Consequence,Exon Number,Intron number,CADD_PHRED,CADD_Prediction,CADD_Grantham,Cancer Hotspot_Gene mutation_Sample frequency,Cancer Hotspot_Gene mutation_Sample frequency-Tumor_type_composition,Cancer Hotspot_Mutation site_Sample frequency,Cancer Hotspot_Mutation site_Sample frequency-Tumor_type_composition,Cancer Hotspot_Exact Mutation_Sample frequency,Intogen_Gene Mutation_Sample frequency,Intogen_Mutation Site_Sample frequency,Intogen_Exact Mutation_Sample frequency,Intogen_Known Driver,OncoKB_link,OncoKB_oncogenic_status,OncoKB_mutation_effect,NCBI Gene ID,CKB Core _Synonyms,CKB Core_GeneRole,CKB Core_Gene_Description
0,1,9665241,A,G,PASS,130,0.0,PIK3CD,,,NaN,NaN,nan,,,,nan,,NaN,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,INTRONIC,NaN,1/23,3.331,Benign,NaN,,,,,,None,None,None,None,https://www.oncokb.org/hgvsg/1:g.9665241A>G?re...,,,5293,APDS | IMD14 | IMD14A | IMD14B | p110D | P110D...,Unknown,"PIK3CD, phosphatidylinositol 4,5-bisphosphate ..."
1,1,9665241,A,G,PASS,130,0.0,PIK3CD,,,NaN,NaN,nan,,,,nan,,NaN,NaN,nan,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UPSTREAM,NaN,NaN,3.331,Benign,NaN,,,,,,,None,None,,https://www.oncokb.org/hgvsg/1:g.9665241A>G?re...,,,5293,APDS | IMD14 | IMD14A | IMD14B | p110D | P110D...,Unknown,"PIK3CD, phosphatidylinositol 4,5-bisphosphate ..."
